# Prática: leitura de hidrômetros (Luminus, Grupo 2)

Notebook fino: toda a lógica está em `src/hidrometros/` (instalável com `pip install -e .`). As saídas são limpas antes do commit (nenhuma imagem do dataset no git).

## 1. Dados

In [ ]:
from hidrometros import data

print(data.check_integrity())
ann = data.load_annotations()
ann.head()

## 2. Preparação: retificação, orientação, 8 recortes de 48x64

In [ ]:
# Gera digits/ e digits/index.csv (local, gitignored). ~3 min.
# !hidrometros prep

## 3. Treino dos quatro modelos (mesmo protocolo)

In [ ]:
# Em segundo plano, com caffeinate; retoma de last.pt com --resume.
# !caffeinate -i hidrometros train all --resume

## 4. Avaliação no teste: results.csv, relatórios, figuras

In [ ]:
# !hidrometros evaluate
# !hidrometros figures
import pandas as pd

pd.read_csv('results.csv')

## 5. Galeria dos 16 erros mais confiantes (apenas local)

A célula abaixo desenha recortes do dataset. Rode localmente e **limpe as saídas antes de commitar** (`jupyter nbconvert --clear-output --inplace`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from hidrometros import data

MODEL = 'resnet18'
p = pd.read_csv(f'reports/predictions_{MODEL}.csv')
idx = pd.read_csv('digits/index.csv')
err = p[p.y_true != p.y_pred].nlargest(16, 'confidence').merge(idx, on=['photo_name', 'position'])
fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for ax, (_, r) in zip(axes.ravel(), err.iterrows()):
    ax.imshow(Image.open(data.ROOT / r.crop_path)); ax.axis('off')
    ax.set_title(f'{r.y_true}->{r.y_pred} ({r.confidence:.2f})', fontsize=8)
plt.show()